# Graphs — User Guide

`StarLayerGraph` extends rdflib's `Graph` to support RDF 1.2: triple terms, reification, and direction-tagged literals.

Two related guides go into more depth: **[2.a Working with datasets](02a-graphs-datasets.ipynb)** (`StarLayerDataset`, multiple named graphs in one data store), **[2.b Serialization formats](02b-graphs-serialization-formats.ipynb)** (support for various RDF 1.2 formats `parse()`/`serialize()` support), and **[2.c Canonical hashing and graph comparison](02c-graphs-canonical-hashing.ipynb)** (RDFC-1.0 canonical hashing/`isomorphic()`, a general graph feature that isn't RDF-1.2-specific but lives in this same package).

One more guide worth knowing about: for SHACL shape validation over the graphs built here, see the [SHACL shapes guide](04-shacl-shapes.ipynb).

Note to claude - still open, not yet done for this guide or any other: standardize the "how to run this notebook" pip-install wording across all guides. Suggested approach - a short pointer instead of repeating the full install block: "See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet." Try it here first, then only propagate to the other 14 guides once the wording is agreed.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. Run cells from top to bottom — later sections reuse variables from earlier ones.

In [1]:
from starlayer import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics

StarLight provides an extension of the rdflib graph model to support RDF 1.2, including:
- Triple terms and statement resources
- Reification via `rdf:reifies` and statement metadata
- Direction-tagged strings such as `"hello"@en--ltr` and `"مرحبا"@ar--rtl`

In [2]:
# create the graph, and assign a namespace
g = StarLayerGraph()
g.bind("ex", EX)

# create a triple term
tt = TripleTerm(EX.bob, EX.knows, EX.carol)

# create a reifier (ex:claim) associated with the triple term and add it to the graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.wikipedia))

print("EX.claim reifies triple term: ")
print((EX.claim, RDF.reifies, tt) in g)
print(g.serialize(format="turtle12"))

EX.claim reifies triple term: 
True
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



In [3]:
g = StarLayerGraph()
g.bind("ex", EX)

# add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.other, RDF.reifies, (EX.bob, EX.likes, EX.dana)))

g.add((EX.bob, EX.knows, EX.dana))

# rdflib triples() now accepts a triple term as the object when selecting triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.bob):
    print(t)

# has_triple_term() tests whether the triple term is in the graph.
# (EX.bob, EX.knows, EX.dana) is asserted directly in the graph, but is not the
# object of any triple, so it returns False.
print(g.has_triple_term(EX.bob, EX.knows, EX.carol))
print(g.has_triple_term(EX.bob, EX.knows, EX.dana))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
True
False


In [4]:
# rdf:reifies is the common approach to making a statement about a statement.
# RDF 1.2 allows triple terms in the object position of any triple.

# add a triple to the graph with a triple term as the object
g.add((EX.dana, EX.said, (EX.bob, EX.knows, EX.carol)))

# triples() accepts a triple term as object to select matching triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

# qname_term() is a starlayer function that adds qname transformation to triple terms too.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
ex:dana ex:said <<( ex:bob ex:knows ex:carol )>>


### Finding what's been said about a statement

`reifiers()`, `reifications()`, `reifier_annotations()`, `reified_triples()`, and `remove_reification()` navigate the reifier/triple-term/annotation relationships directly, without SPARQL queries. `remove_reification(reifier, triple_term=None)` can be scoped to one specific reifier↔triple link, leaving any other triple(s) the same reifier reifies — and all its annotations — untouched; omit `triple_term` for the original all-or-nothing behavior.

In [5]:
# continues using g from the previous cell

# adds assertions to the reification ex:claim
g.add((EX.claim, EX.source, EX.wikipedia))

tt1 = (EX.bob, EX.knows, EX.carol)
tt2 = (EX.bob, EX.likes, EX.dana)

# reifiers(): returns the reifier node(s) that reify a given triple term.
print([g.qname(r) for r in g.reifiers(TT=tt1)])

# reifications(): returns a list of triple terms that have at least one reifier.
for tt in g.reifications():
    print(tt)

# reifier_annotations(): returns a reifier's annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname(reifier), g.qname(pred), g.qname(val))

# reified_triples(): returns the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(tt)

# give ex:claim a second rdf:reifies link, so scoped vs. wildcard removal are distinguishable
g.add((EX.claim, RDF.reifies, tt2))

# remove_reification(reifier, triple_term): remove the reification link between a reifier
# and the specified triple term only.
g.remove_reification(EX.claim, tt1)
print((EX.claim, RDF.reifies, tt1) in g)         # False - only this link removed
print((EX.claim, RDF.reifies, tt2) in g)         # True  - untouched
print((EX.claim, EX.source, EX.wikipedia) in g)  # True  - untouched

# remove_reification(reifier): removes all rdf:reifies links from the reifier
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt2) in g)
print((EX.claim, EX.source, EX.wikipedia) in g)

['ex:claim']
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
ex:claim ex:source ex:wikipedia
<<( ex:bob ex:knows ex:carol )>>
False
True
True
False
True


### Direction-tagged string literals

`DirLangString` sets the base direction of a language-tagged string literal.

In [6]:
g = StarLayerGraph()
g.bind("ex", EX)

# literals can now include language direction
g.add((EX.title, EX.value, DirLangString("مرحبا", "ar", "rtl")))
g.add((EX.title, EX.value, Literal("hello", "en")))
g.add((EX.title, EX.value, DirLangString("hello", "en", "ltr")))

print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:title ex:value "hello"@en, "مرحبا"@ar--rtl, "hello"@en--ltr .



## Further work

See [Working with datasets](02a-graphs-datasets.ipynb)'s, [Serialization formats](02b-graphs-serialization-formats.ipynb)'s, and [Canonical hashing and graph comparison](02c-graphs-canonical-hashing.ipynb)'s own Further Work sections for open items specific to those topics.